In [1]:
from pymargo.core import Engine
import pyyokan_common as yokan
from pyyokan_client import Client
from pyyokan_server import Provider

In [2]:
import json
import ctypes
import struct
import blosc2
import numpy as np

In [3]:
f = open('/projects/insituperf/seer_o/test_app/mochi-yokan-config.json')
json_data = json.load(f)
json_data

{'sim-id': '07251202',
 'libraries': {'yokan': '/vast/home/pascalgrosset/spack/opt/spack/linux-rhel8-haswell/gcc-9.4.0/mochi-yokan-0.4.2-hiu7yh7om6nmyc2ahuknpdsov5k64zcj/lib/libyokan-bedrock-module.so'},
 'providers': [{'name': 'yokan_provider',
   'provider_id': 124,
   'type': 'yokan',
   'pool': '__primary__',
   'config': {'database': {'type': 'map'}}}],
 'data': [{'name': 'pressure_3', 'compressor': 'BLOSC'},
  {'name': 'temperature_3', 'compressor': 'SZ3', 'psnr': 50}],
 'databases': [{'address': '192.168.81.72:42143',
   'protocol': 'ofi+tcp',
   'provider_id': 124}]}

In [ ]:
json_data['providers'][0]['config']

In [ ]:
dbs = []
for db in json_data['databases']:
    address = db['address']
    protocol = db['protocol']
    provider_id = db['provider_id']
    server_addr = protocol + '://' + address
    print(server_addr)
    
    engine = Engine(protocol)
    mid = engine.get_internal_mid()
    addr = engine.lookup(server_addr)
    hg_addr = addr.get_internal_hg_addr()
    provider = Provider(mid=mid, provider_id=provider_id, config='{"database":{"type":"map"}}')
    client = Client(mid=mid)
    db = client.make_database_handle(address=hg_addr, provider_id=provider_id)
    
    dbs.append(db)
    
   

In [6]:
server_addr1 = "ofi+tcp://192.168.81.72:42143"
server_addr2 = "ofi+tcp://192.168.81.100:33977"
provider_id = 124
protocol = 'ofi+tcp'

In [7]:
engine1 = Engine(protocol)
mid1 = engine1.get_internal_mid()
addr1 = engine1.lookup(server_addr1)
hg_addr1 = addr1.get_internal_hg_addr()
provider1 = Provider(mid=mid1, provider_id=provider_id, config='{"database":{"type":"map"}}')
client1 = Client(mid=mid1)
db1 = client1.make_database_handle(address=hg_addr1, provider_id=provider_id)

In [ ]:
engine2 = Engine(protocol)
mid2 = engine2.get_internal_mid()
addr2 = engine2.lookup(server_addr2)
hg_addr2 = addr2.get_internal_hg_addr()
provider2 = Provider(mid=mid2, provider_id=provider_id, config='{"database":{"type":"map"}}')
client2 = Client(mid=mid2)
db2 = client2.make_database_handle(address=hg_addr2, provider_id=provider_id)

In [8]:
dbs = []

In [9]:
dbs.append(db1)
#dbs.append(db2)

In [ ]:
# Params
provider_id = 123
#protocol = 'na+sm'
protocol = 'ofi+tcp'
#server_addr = 'na+sm://1364645-0'
server_addr = 'ofi+tcp://192.168.81.80:38989'

In [ ]:
engine = Engine(protocol)
mid = engine.get_internal_mid()
addr = engine.lookup(server_addr)
hg_addr = addr.get_internal_hg_addr()
provider = Provider(mid=mid, provider_id=provider_id, config='{"database":{"type":"map"}}')
client = Client(mid=mid)
db = client.make_database_handle(address=hg_addr, provider_id=provider_id)

In [10]:
ts = '_2'

In [11]:
def list_all_keys(db):
    num_keys = db.count()
    
    max_length = 1024
    prefix = ''
    out_keys = []
    for i in range(0, num_keys):
      out_keys.append( bytearray(max_length+len(prefix)+1) )
    
    from_key = ''
    ksizes = db.list_keys(keys=out_keys, from_key=from_key, filter=prefix)
    
    keys = []
    for i in range(len(ksizes)):
        key_size = ksizes[i]
        
        k = out_keys[i]
        key = (k[:key_size]).decode('ascii')
        keys.append(key)
        
    return keys

In [12]:
def split_key(key, pos):
    parts = key.split('/')
    name = parts[pos]
    return name

In [13]:
def get_field_name(key):
    parts = keys[0].split('/')
    name = parts[len(parts)-2]
    return name

In [14]:
def list_fields(db, timestep):
    
    all_keys = list_all_keys(db)
    print("all_keys:", all_keys)
    
    x = []
    for k in all_keys:
        parts = k.split('/')
        name = parts[len(parts)-2]
        x.append(name)
    return list(set(x))

In [15]:
def list_attributes(db, key):
    all_keys = list_all_keys(db)

    x = []
    for k in all_keys:
        parts = k.split('/')
        name = parts[2]
        field = parts[3]
        if name == key:
            x.append(field)
    x = list(set(x))

    return x

In [16]:
def get_value(db, key):
    ''' Get data from the server for that key '''

    # length of the value associated with the key
    l = db.length(key)

    out_val = bytearray(l)          # create buffer
    db.get(key=key, value=out_val)  # get the data
    v = out_val.decode("ascii")     # convert to ascii
    return v

In [17]:
def get_data(db, key):
    ''' Get data from the server for that key '''

    # length of the value associated with the key
    l = db.length(key)

    out_val = bytearray(l)          # create buffer
    db.get(key=key, value=out_val)  # get the data
    return out_val

In [ ]:
def get_decompDataBLOSC(db, key):
        num_elems = 21360
        n_e = str(num_elems) + 'f'
        
        x = []
        val = get_data(db, key)
        a_bytesobj2 = blosc2.decompress(val)
        buf_size = len(a_bytesobj2)
        #x = struct.unpack(n_e, a_bytesobj2)
        x = struct.unpack(buf_size, a_bytesobj2)

        return x

In [ ]:
import struct

In [ ]:
keys = list_fields(dbs[0], ts)
keys

In [ ]:
keys2 = list_fields(dbs[1], ts)
keys2

In [ ]:
get_value(dbs[0],"_12345_499/2/x/num_elems")

In [ ]:
com_x = get_data(dbs[0], "_12345_499/2/x/value")

In [ ]:
a_bytesobj2 = blosc2.decompress(com_x)

In [ ]:
len(a_bytesobj2)

In [ ]:
len(com_x)

In [ ]:
vals_x = get_decompData(dbs[0], "_12345_499/2/x/value")
vals_y = get_decompData(dbs[0], "_12345_499/2/y/value")
vals_z = get_decompData(dbs[0], "_12345_499/2/z/value")

In [ ]:
len(vals_x)

In [ ]:
len(vals_y)

In [ ]:
len(vals_z)

In [ ]:
with open('output.csv', 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)

    # Write each array as a row
    writer.writerow(vals_x)
    writer.writerow(vals_y)
    writer.writerow(vals_z)

In [ ]:
xxx= np.stack([vals_x,vals_y,vals_z], axis=1)

In [ ]:
xxx

In [ ]:
np.savetxt("3d_array.csv", xxx, delimiter=",")